In [1]:
"""
====================================================================
STUDY B ONLY -- CROSS-DATASET GENERALIZATION PROBE, ALL 7 MODELS
Tissue-Structure Reconstruction for Cross-Centre Generalization
(PID 23 -- Achanta Badrinath / Peesari Sathvik Reddy)

WHAT THIS RUNS
--------------------------------------------------------------------
Just Study B, split out of probe_studies_full_combined.py so it can be
re-run on its own without re-running Study A (the probe battery: centre/
position/patient probes) every time -- the two studies don't share any
state besides which checkpoints/model to load, so there was never a
reason to force them to run together.

STUDY B -- CROSS-DATASET GENERALIZATION: a tumor probe is fit ONCE on
Camelyon17-WILDS features (logistic regression, class_weight='balanced'),
evaluated in-domain (held-out Camelyon17-WILDS patients), then evaluated
zero-shot (the SAME fitted probe, never refit) on THREE external target
datasets of increasing domain distance:

  1. PatchCamelyon      -- same organ (breast lymph-node), disjoint
                            hospitals -- cross-institution shift.
  2. NCT-CRC-HE-100K     -- colorectal tissue, tumor epithelium vs. 8
                            non-tumor classes -- cross-organ shift.
  3. LC25000             -- lung/colon tissue, malignant vs. benign --
                            cross-organ shift, different cancer types.

Reports, for every one of the 7 models: in-domain balanced-accuracy
(the ceiling), each target's balanced-accuracy/AUROC, and the
generalization gap (in-domain minus target) for each target.

All 7 models covered (identical loading/adapter code to the combined
probe script and the combined XAI notebook):
  - recon_resnet18, recon_vit_small, recon_swin_tiny  (windowed
    reconstruction backbones -- real cross-patch context-transformer)
  - baseline_stain_aug                                 (patch-independent,
    own trained head, per-LOCO-fold checkpoint)
  - kaiko_vits16, lunit_vits16, phikon                  (patch-independent
    frozen pathology foundation models)

For Study B specifically there is NO architectural asymmetry to worry
about: every model reduces to "one patch in, one embedding out" under
the masked single-patch protocol (see extract_masked_embeddings below),
so all 7 numbers are a completely fair, apples-to-apples comparison --
unlike Study A's probes 2/4, this one needed no special-casing.

CONFIG
--------------------------------------------------------------------
  - 3 reconstruction backbones: loaded from the SAME kagglehub dataset
    used everywhere else in this project
    ("peesarisathvikreddy/histopathology-reconstruction-checkpoints").
  - baseline_stain_aug: loaded from the SAME per-fold checkpoints
    dataset used by the combined XAI notebook ("setu295452/stain-aug"),
    using the TEST_CENTER=2 held-out fold (the "primary" split, same
    convention as every other script in this project).
  - kaiko_vits16 / phikon / lunit_vits16: loaded directly from the
    Hugging Face Hub (open weights, no login needed).

Run:
    python study_b_cross_dataset_standalone.py
====================================================================
"""

import os
import random
import gc
import glob
import warnings

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score

warnings.filterwarnings('ignore')


# ====================================================================
# CONFIG
# ====================================================================
class Config:
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"

    # ---- Study B target datasets -- THREE external datasets of increasing domain
    # distance from Camelyon17-WILDS. `positive_keywords`: substrings
    # (case-insensitive) matched against the dataset's class NAMES to decide which
    # classes count as "tumor/malignant" -- None means the label column is already
    # binary (PCam's True/False), so no keyword matching needed.
    TARGET_DATASETS = [
        dict(name="PatchCamelyon", hf_id="1aurent/PatchCamelyon", split="test",
             label_col="label", positive_keywords=None, n_per_class=3000,
             note="same organ (breast lymph-node), disjoint hospitals -- cross-institution shift"),
        dict(name="NCT-CRC-HE-100K", hf_id="1aurent/NCT-CRC-HE", split="CRC_VAL_HE_7K",
             label_col="label", positive_keywords=["tum"], n_per_class=800,
             note="colorectal tissue, tumor epithelium vs. 8 non-tumor classes -- cross-organ shift"),
        dict(name="LC25000", hf_id="1aurent/LC25000", split="train",
             # actual class names are ['benign', 'adenocarcinomas', 'squamous carcinomas']
             # (confirmed by a real run's label diagnostic) -- not 'lung_aca'/'colon_aca'-style
             # short codes, which matched nothing.
             label_col="label", positive_keywords=["adenocarcinom", "squamous"], n_per_class=2000,
             note="lung/colon tissue, malignant vs. benign -- cross-organ shift, different cancer types"),
    ]

    RECON_KAGGLEHUB_DATASET = "peesarisathvikreddy/histopathology-reconstruction-checkpoints"
    RECON_CHECKPOINT_DIR = "/kaggle/input/datasets/peesarisathvikreddy/histopathology-reconstruction-checkpoints"
    RECON_CHECKPOINT_MATCH = {
        'recon_resnet18':  'resnet18',
        'recon_vit_small': 'vit_small',
        'recon_swin_tiny': 'swin_tiny',
    }
    RECON_BACKBONE_OF = {   # maps our model name -> the underlying architecture name
        'recon_resnet18':  'resnet18',
        'recon_vit_small': 'vit_small',
        'recon_swin_tiny': 'swin_tiny',
    }
    RECON_BACKBONE_CONFIGS = {
        'resnet18':  dict(dropout=0.3, freeze_stages=2, context_layers=4),
        'vit_small': dict(dropout=0.1, freeze_stages=0, context_layers=4),
        'swin_tiny': dict(dropout=0.1, freeze_stages=0, context_layers=3),
    }

    STAIN_AUG_KAGGLEHUB_DATASET = "setu295452/stain-aug"
    STAIN_AUG_INPUT_DIR = "/kaggle/input/stain-aug"
    STAIN_AUG_TEST_CENTER = 2   # the "primary" fold -- same convention as every other script here
    STAIN_AUG_INPUT_SIZE = 96

    FOUNDATION_MODELS = {
        'kaiko_vits16': {
            'hub_id': 'hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms',
            'loader': 'timm', 'input_size': 224,
        },
        'phikon': {
            'hub_id': 'owkin/phikon',
            'loader': 'transformers', 'input_size': 224,
        },
        'lunit_vits16': {
            'hub_id': 'hf-hub:1aurent/vit_small_patch16_224.lunit_dino',
            'loader': 'timm', 'input_size': 224,
        },
    }

    # Row order for every combined table/figure below: reconstruction backbones
    # first, then the stain-aug baseline, then the 3 foundation models.
    MODEL_ORDER = ['recon_resnet18', 'recon_vit_small', 'recon_swin_tiny',
                   'baseline_stain_aug', 'kaiko_vits16', 'lunit_vits16', 'phikon']
    WINDOWED_MODELS = {'recon_resnet18', 'recon_vit_small', 'recon_swin_tiny'}   # true context-transformer models
    PATCH_LEVEL_MODELS = {'baseline_stain_aug', 'kaiko_vits16', 'lunit_vits16', 'phikon'}

    NUM_POSITIONS = 9
    NUM_PERMUTATIONS = 500     # unused by this probe, kept only because ReconstructionModel's
                               # __init__ signature expects it (perm_head is never called here)
    CNN_INPUT_SIZE = 96
    VIT_INPUT_SIZE = 224

    ALL_CENTERS = [0, 1, 2, 3, 4]
    VAL_PATIENT_FRAC = 0.3
    SEED = 42

    # ---- Study B (cross-dataset) sample sizes ----
    CAMELYON17_LABEL_COLUMN = "label"
    CAMELYON17_TRAIN_N_PER_CLASS = 3000
    CAMELYON17_TEST_N_PER_CLASS = 1000
    LABEL_CANDIDATE_KEYWORDS = ("tumor", "label", "target", "is_tumor", "y")
    SKEW_OVERRIDE_THRESHOLD = 0.97
    BALANCE_DIAGNOSTIC_SAMPLE = 20000
    MIN_CLASS_FOR_RELIABLE_AUROC = 30

    NUM_WORKERS = 4
    RESULTS_DIR = "/kaggle/working/probe_results"


os.makedirs(Config.RESULTS_DIR, exist_ok=True)


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def get_input_size(name):
    if name in Config.WINDOWED_MODELS:
        backbone = Config.RECON_BACKBONE_OF[name]
        return Config.VIT_INPUT_SIZE if backbone in ('vit_small', 'swin_tiny') else Config.CNN_INPUT_SIZE
    if name == 'baseline_stain_aug':
        return Config.STAIN_AUG_INPUT_SIZE
    return Config.FOUNDATION_MODELS[name]['input_size']


def build_transform(name):
    size = get_input_size(name)
    norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    return transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(), norm])


# ====================================================================
# MODEL A -- the windowed ReconstructionModel (recon_resnet18/vit_small/
# swin_tiny). Real cross-patch mixing via a transformer encoder over the
# 9 slots, positional slot_embed, and a learned mask_token for "unknown
# neighbour" -- identical architecture/loading to every other probe
# script in this project.
# ====================================================================
class ReconstructionModel(nn.Module):
    def __init__(self, backbone_name, num_positions=9, num_permutations=500,
                 dropout=0.1, freeze_stages=0, context_layers=4):
        super().__init__()
        self.backbone_name = backbone_name
        self.num_positions = num_positions
        self.input_size = get_input_size(
            {'resnet18': 'recon_resnet18', 'vit_small': 'recon_vit_small',
             'swin_tiny': 'recon_swin_tiny'}[backbone_name])
        self._build_backbone(backbone_name)
        self.proj = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.LayerNorm(self.feature_dim),
        )
        self.context = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=self.feature_dim, nhead=8,
                dim_feedforward=min(4 * self.feature_dim, 2048),
                dropout=dropout, batch_first=True, norm_first=True),
            num_layers=context_layers)
        self.slot_embed = nn.Parameter(torch.zeros(1, num_positions, self.feature_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, self.feature_dim))
        # perm_head/position_head exist in the trained checkpoint's state_dict
        # (so load_state_dict has somewhere to put those weights) but are never
        # called by this probe -- Study B only reads off get_embedding()'s
        # context-transformer output directly.
        self.perm_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim * num_positions, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_permutations))
        self.position_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_positions))

    def _build_backbone(self, name):
        if name == 'resnet18':
            net = tvm.resnet18(weights=None)
            self.feature_dim = 512
            self.backbone = nn.Sequential(*list(net.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d(1)
        elif name == 'vit_small':
            import timm
            self.backbone = timm.create_model(
                'vit_small_patch16_224', pretrained=False,
                num_classes=0, img_size=self.input_size)
            self.feature_dim, self.pool = 384, None
        elif name == 'swin_tiny':
            import timm
            self.backbone = timm.create_model(
                'swin_tiny_patch4_window7_224', pretrained=False,
                num_classes=0, img_size=self.input_size)
            self.feature_dim, self.pool = 768, None
        else:
            raise ValueError(f"Backbone {name} not supported in this file")

    def _encode(self, x, mask=None):
        B, N = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        f = self.proj(f).view(B, N, self.feature_dim)
        if mask is not None and mask.any():
            f = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(f), f)
        return self.context(f + self.slot_embed)

    @torch.no_grad()
    def get_embedding(self, x, mask=None):
        return self._encode(x, mask=mask)


def _resolve_recon_checkpoint_dir():
    alt_paths = [
        Config.RECON_CHECKPOINT_DIR,
        "/kaggle/input/histopathology-reconstruction-checkpoints",
    ]
    for p in alt_paths:
        if os.path.isdir(p):
            print(f"OK found recon checkpoints at classic input mount {p}")
            return p
    import kagglehub
    path = kagglehub.dataset_download(Config.RECON_KAGGLEHUB_DATASET)
    print(f"OK resolved recon checkpoints via kagglehub -> {path}")
    return path


def _find_recon_checkpoints(root):
    all_pt = glob.glob(os.path.join(root, '**', '*.pt'), recursive=True)
    found = {}
    for name, needle in Config.RECON_CHECKPOINT_MATCH.items():
        cand = [p for p in all_pt if needle in os.path.basename(p).lower()]
        if cand:
            found[name] = cand[0]
        else:
            print(f"FLAG no checkpoint matched for '{name}' (looked for '*{needle}*.pt' under {root})")
    return found


def load_recon_model(name, ckpt_path, device):
    backbone = Config.RECON_BACKBONE_OF[name]
    cfg = Config.RECON_BACKBONE_CONFIGS[backbone]
    model = ReconstructionModel(
        backbone, Config.NUM_POSITIONS, Config.NUM_PERMUTATIONS,
        dropout=cfg['dropout'], freeze_stages=cfg['freeze_stages'],
        context_layers=cfg['context_layers'],
    ).to(device)
    # weights_only=False: PyTorch 2.6 flipped torch.load's default to True, which
    # rejects the plain numpy scalars (e.g. best_pos) these checkpoints also carry
    # alongside the state_dict ("Unsupported global: numpy._core.multiarray.scalar").
    # These are our own checkpoints, so this is safe.
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ck['model_state_dict'] if isinstance(ck, dict) and 'model_state_dict' in ck else ck
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"FLAG [{name}] load_state_dict mismatches -- missing={len(missing)} unexpected={len(unexpected)}")
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    print(f"OK [{name}] loaded {ckpt_path}")
    model.feature_dim_out = model.feature_dim
    return model


# ====================================================================
# MODEL B -- PATCH-LEVEL ADAPTER (baseline_stain_aug + the 3 foundation
# models). These process ONE patch at a time, independently -- no
# cross-patch mixing exists, so `get_embedding` on a window just encodes
# each of the 9 real patches on its own, and under the masked-single-
# patch protocol (mask is not None) it short-circuits to encoding ONLY
# slot 0 (the one real patch) -- Study B's extract_masked_embeddings
# below only ever reads back slot 0 anyway, so skipping slots 1-8
# entirely is a pure speed win, not an approximation.
# ====================================================================
class PatchLevelAdapter(nn.Module):
    def __init__(self, encoder, feature_dim):
        super().__init__()
        self.encoder = encoder      # any nn.Module exposing .encode_single(x) -> (B, D)
        self.feature_dim = feature_dim

    @torch.no_grad()
    def get_embedding(self, x, mask=None):
        B, N = x.shape[:2]
        if mask is not None and mask.any():
            real = x[:, 0]                                  # (B, C, H, W) -- slot 0 only
            f0 = self.encoder.encode_single(real)
            out = torch.zeros(B, N, self.feature_dim, device=x.device, dtype=f0.dtype)
            out[:, 0] = f0
            return out
        flat = x.flatten(0, 1)                               # (B*N, C, H, W)
        f = self.encoder.encode_single(flat)
        return f.view(B, N, self.feature_dim)


def build_stain_aug_classifier():
    """Identical architecture to the combined XAI notebook's version -- an
    ImageNet-initialized resnet18 with a fresh binary head, trained end-to-end."""
    net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
    net.fc = nn.Linear(net.fc.in_features, 1)
    return net


class _StainAugEncoder(nn.Module):
    """Strips the trained classification head off a loaded baseline_stain_aug
    checkpoint, keeping everything through avgpool -- i.e. the same 512-dim
    penultimate representation every other probe in this project reads off
    a resnet18-family model."""
    def __init__(self, stain_aug_model):
        super().__init__()
        self.backbone = nn.Sequential(*list(stain_aug_model.children())[:-1])   # drops fc only

    @torch.no_grad()
    def encode_single(self, x):
        f = self.backbone(x)
        return torch.flatten(f, 1)


def _resolve_stain_aug_checkpoint():
    alt_paths = [Config.STAIN_AUG_INPUT_DIR]
    root = None
    for p in alt_paths:
        if os.path.isdir(p):
            root = p
            print(f"OK found stain-aug checkpoints at classic input mount {p}")
            break
    if root is None:
        import kagglehub
        root = kagglehub.dataset_download(Config.STAIN_AUG_KAGGLEHUB_DATASET)
        print(f"OK resolved stain-aug checkpoints via kagglehub -> {root}")

    target = f"stain_aug_holdout{Config.STAIN_AUG_TEST_CENTER}.pt"
    matches = glob.glob(os.path.join(root, '**', target), recursive=True)
    if not matches:
        raise FileNotFoundError(f"could not find {target} under {root}")
    return matches[0]


def load_stain_aug_adapter(device):
    ckpt_path = _resolve_stain_aug_checkpoint()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)   # see FIX note above
    net = build_stain_aug_classifier().to(device)
    net.load_state_dict(ckpt['state_dict'])
    net.eval()
    for p in net.parameters():
        p.requires_grad = False
    print(f"OK [baseline_stain_aug] loaded {ckpt_path} "
          f"(trained on centres {ckpt['train_centers']}, primary AUROC={ckpt['auroc']:.4f})")
    adapter = PatchLevelAdapter(_StainAugEncoder(net).to(device), feature_dim=512)
    return adapter


class _TimmViTEncoder(nn.Module):
    def __init__(self, hub_id):
        super().__init__()
        import timm as _timm
        self.backbone = _timm.create_model(hub_id, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, 'num_features', None) or 768

    @torch.no_grad()
    def encode_single(self, x):
        f = self.backbone(x)
        return f if f.dim() == 2 else f.mean(1)


class _HFTransformersViTEncoder(nn.Module):
    def __init__(self, hub_id):
        super().__init__()
        from transformers import AutoModel
        self.backbone = AutoModel.from_pretrained(hub_id, attn_implementation="eager")
        self.feature_dim = self.backbone.config.hidden_size

    @torch.no_grad()
    def encode_single(self, x):
        out = self.backbone(pixel_values=x)
        return out.last_hidden_state[:, 0]   # CLS token


def load_foundation_adapter(name, device):
    cfg = Config.FOUNDATION_MODELS[name]
    if cfg['loader'] == 'transformers':
        enc = _HFTransformersViTEncoder(cfg['hub_id']).to(device)
    else:
        enc = _TimmViTEncoder(cfg['hub_id']).to(device)
    enc.eval()
    for p in enc.parameters():
        p.requires_grad = False
    print(f"OK [{name}] loaded {cfg['hub_id']} (dim={enc.feature_dim})")
    return PatchLevelAdapter(enc, feature_dim=enc.feature_dim)


# ====================================================================
# UNIFIED MODEL LOADER -- returns an object exposing .get_embedding(x,
# mask=None) and .feature_dim, regardless of which of the two model
# families (windowed vs. patch-level) it actually is.
# ====================================================================
def load_model_for_probes(name, device, recon_ckpts):
    if name in Config.WINDOWED_MODELS:
        return load_recon_model(name, recon_ckpts[name], device)
    if name == 'baseline_stain_aug':
        return load_stain_aug_adapter(device)
    if name in Config.FOUNDATION_MODELS:
        return load_foundation_adapter(name, device)
    raise ValueError(f"no loader registered for {name}")


# ====================================================================
# DATA LOADING
# ====================================================================
def load_camelyon17():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Config.HF_DATASET)
    parts = [ds[k] for k in ds.keys()]
    print(f"[Camelyon17-WILDS] splits: {', '.join(f'{k}={len(ds[k]):,}' for k in ds.keys())}")
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"OK [Camelyon17-WILDS] concatenated: {len(full):,} patches")
    return full


def load_target_dataset(spec):
    """Generic loader for any of Config.TARGET_DATASETS -- pulls the configured
    split from the configured HF dataset id."""
    from datasets import load_dataset
    ds_all = load_dataset(spec['hf_id'])
    print(f"[{spec['name']}] splits: {', '.join(f'{k}={len(ds_all[k]):,}' for k in ds_all.keys())}")
    split = spec['split']
    if split not in ds_all:
        raise ValueError(f"'{split}' split not found in {spec['hf_id']}; available: {list(ds_all.keys())}")
    ds = ds_all[split]
    print(f"OK [{spec['name']}] using split '{split}': {len(ds):,} patches")
    return ds


def resolve_binary_label(dataset, label_col, positive_keywords, dataset_name):
    """Generic binary-label resolution for Study B target datasets: handles both
    an already-binary column (PCam's True/False) and a multi-class ClassLabel
    column (NCT-CRC-HE's 9 classes, LC25000's 3 classes) by matching class NAMES
    against keywords -- never by assuming a particular integer code, which is
    exactly the kind of assumption that caused the earlier Camelyon17 tumor-label
    bug. Always prints the resolved name->binary mapping so a wrong keyword match
    is visible immediately instead of silently producing a skewed pool."""
    print(f"\n--- LABEL DIAGNOSTIC -- {dataset_name} ---")
    feature = dataset.features[label_col]
    names = getattr(feature, 'names', None)

    if positive_keywords is None or names is None:
        raw = np.asarray(dataset[label_col])
        labels = raw.astype(np.int64)
        uniq = sorted(np.unique(labels).tolist())
        pos_rate = float((labels == uniq[-1]).mean()) if len(uniq) == 2 else float('nan')
        print(f"column '{label_col}' is already binary: values={uniq}, positive-rate={pos_rate:.4f}")
        return labels

    matched = [n for n in names if any(kw.lower() in n.lower() for kw in positive_keywords)]
    unmatched = [n for n in names if n not in matched]
    print(f"class names: {names}")
    print(f"matched -> POSITIVE(1): {matched}")
    print(f"unmatched -> NEGATIVE(0): {unmatched}")
    if not matched:
        raise ValueError(f"No class name in {names} matched keywords {positive_keywords} for "
                          f"{dataset_name} -- fix Config.TARGET_DATASETS['positive_keywords'].")

    idx_to_binary = np.array([1 if n in matched else 0 for n in names], dtype=np.int64)
    raw_idx = np.asarray(dataset[label_col], dtype=np.int64)
    labels = idx_to_binary[raw_idx]
    pos_rate = float(labels.mean())
    print(f"resulting positive rate: {pos_rate:.4f}  ({int(labels.sum())} pos / "
          f"{int((1 - labels).sum())} neg out of {len(labels)})")
    return labels


def resolve_label_column(dataset, dataset_name, configured_col):
    """Shared label-column auto-detection: checks that the configured label
    column isn't severely skewed before trusting it."""
    non_image_cols = [c for c in dataset.column_names if c != 'image']
    candidates = [c for c in non_image_cols
                  if any(k in c.lower() for k in Config.LABEL_CANDIDATE_KEYWORDS)]
    if configured_col not in candidates and configured_col in non_image_cols:
        candidates.append(configured_col)

    print(f"\n--- LABEL COLUMN DIAGNOSTIC -- {dataset_name} ---")
    print(f"candidates: {candidates or '(none -- keeping configured default)'}")
    if not candidates:
        return configured_col

    n = min(Config.BALANCE_DIAGNOSTIC_SAMPLE, len(dataset))
    rng = np.random.RandomState(Config.SEED)
    idx = sorted(rng.choice(len(dataset), size=n, replace=False).tolist())
    sub = dataset.select(idx).select_columns(candidates).to_pandas()

    stats = {}
    for col in candidates:
        vals = sub[col]
        uniq = sorted(vals.unique().tolist())
        if len(uniq) != 2:
            continue
        pos_rate = float((vals == uniq[-1]).mean())
        skew = max(pos_rate, 1 - pos_rate)
        stats[col] = skew
        flag = "  <-- SEVERELY SKEWED" if skew > Config.SKEW_OVERRIDE_THRESHOLD else ""
        print(f"   '{col}': positive-rate={pos_rate:.4f} (skew={skew:.4f}){flag}")

    if not stats:
        return configured_col
    configured_skew = stats.get(configured_col, 1.0)
    best_col = min(stats, key=stats.get)
    best_skew = stats[best_col]
    if configured_skew > Config.SKEW_OVERRIDE_THRESHOLD and best_skew < configured_skew:
        print(f"FLAG switching label column '{configured_col}' -> '{best_col}'")
        return best_col
    return configured_col


def build_patient_split_by_centre(camelyon17_meta, val_frac, seed):
    """Per-centre, patient-level split -- identical convention to every other
    script in this project, so numbers here are directly comparable."""
    train_pats, test_pats = {}, {}
    for centre in Config.ALL_CENTERS:
        pats = sorted(camelyon17_meta.loc[camelyon17_meta['center'] == centre, 'patient'].unique().tolist())
        rng_c = random.Random(seed + centre)
        rng_c.shuffle(pats)
        if len(pats) < 2:
            train_pats[centre], test_pats[centre] = set(pats), set()
            continue
        n_test = max(1, int(round(val_frac * len(pats))))
        n_test = min(n_test, len(pats) - 1)
        test_pats[centre] = set(pats[:n_test])
        train_pats[centre] = set(pats[n_test:])
    return train_pats, test_pats


# ====================================================================
# STUDY B -- CROSS-DATASET GENERALIZATION (Camelyon17-WILDS -> 3 external
# datasets: PatchCamelyon, NCT-CRC-HE-100K, LC25000)
# ====================================================================
class SinglePatchDataset(Dataset):
    """One standalone patch + its (already-resolved binary) label. For
    recon/windowed models this deliberately IGNORES real neighbours (masked-
    single-patch protocol, via extract_masked_embeddings below); for patch-
    level models it's simply what they always do -- one patch in, one
    embedding out. Works identically for Camelyon17-WILDS and every target
    dataset (none of which have neighbour/coordinate metadata at all).
    Takes a pre-resolved `labels` array (indexed the same way as `indices`,
    i.e. by absolute row position) rather than a label COLUMN name, since
    the multi-class target datasets (NCT-CRC-HE, LC25000) need their labels
    binarized by resolve_binary_label() before this ever sees them -- a raw
    per-row column lookup would return the wrong (multi-class) value."""
    def __init__(self, hf_dataset, indices, labels, transform):
        self.ds = hf_dataset
        self.indices = indices
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        row = self.ds[idx]
        img = row['image']
        img = img.convert('RGB') if img.mode != 'RGB' else img
        return {'image': self.transform(img), 'label': int(self.labels[idx])}


def collate_single(batch):
    images = torch.stack([b['image'] for b in batch])
    labels = torch.tensor([b['label'] for b in batch], dtype=torch.long)
    return {'image': images, 'label': labels}


def stratified_sample_indices(labels, n_per_class, seed):
    rng = np.random.RandomState(seed)
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]
    rng.shuffle(pos_idx); rng.shuffle(neg_idx)
    k = min(n_per_class, len(pos_idx), len(neg_idx))
    if k < n_per_class:
        print(f"   FLAG only {len(pos_idx)} pos / {len(neg_idx)} neg available -- using {k}/class")
    idx = np.concatenate([pos_idx[:k], neg_idx[:k]])
    rng.shuffle(idx)
    return idx


@torch.no_grad()
def extract_masked_embeddings(model, loader, device):
    """MASKED SINGLE-PATCH PROTOCOL: every patch, from EITHER dataset, is
    embedded via the exact same procedure, so a score difference reflects
    dataset shift, not 'windowed context vs. no context'. For a windowed
    model this genuinely masks 8 of the 9 slots with its own mask_token;
    for a PatchLevelAdapter, get_embedding already short-circuits to
    encoding only the one real patch (see PatchLevelAdapter docstring)."""
    embs, labels = [], []
    for batch in tqdm(loader, desc="extract(masked)", leave=False):
        x = batch['image'].to(device, non_blocking=True)       # (B, 3, H, W)
        B = x.shape[0]
        x_win = x.unsqueeze(1).expand(B, Config.NUM_POSITIONS, *x.shape[1:]).contiguous()
        mask = torch.zeros(B, Config.NUM_POSITIONS, dtype=torch.bool, device=device)
        mask[:, 1:] = True
        ctx = model.get_embedding(x_win, mask=mask)             # (B, 9, D)
        embs.append(ctx[:, 0, :].cpu().numpy())
        labels.append(batch['label'].numpy())
    return np.concatenate(embs), np.concatenate(labels)


def tumor_probe_fit(X_train, y_train, X_test, y_test, max_iter=1000):
    clf = LogisticRegression(max_iter=max_iter, class_weight='balanced')
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, pred) * 100
    bal_acc = balanced_accuracy_score(y_test, pred) * 100
    n_errors = int((pred != y_test).sum())
    n_test_pos = int(np.sum(y_test == 1))
    n_test_neg = int(np.sum(y_test == 0))
    auroc_reliable = min(n_test_pos, n_test_neg) >= Config.MIN_CLASS_FOR_RELIABLE_AUROC
    try:
        auroc = roc_auc_score(y_test, proba)
    except ValueError:
        auroc = float('nan')
    return {
        'acc': acc, 'balanced_acc': bal_acc, 'auroc': auroc, 'auroc_reliable': auroc_reliable,
        'n_errors': n_errors, 'n_test': len(y_test),
        'n_test_pos': n_test_pos, 'n_test_neg': n_test_neg,
    }, clf


def tumor_probe_eval(clf, X, y):
    pred = clf.predict(X)
    proba = clf.predict_proba(X)[:, 1]
    acc = accuracy_score(y, pred) * 100
    bal_acc = balanced_accuracy_score(y, pred) * 100
    n_errors = int((pred != y).sum())
    n_pos = int(np.sum(y == 1)); n_neg = int(np.sum(y == 0))
    auroc_reliable = min(n_pos, n_neg) >= Config.MIN_CLASS_FOR_RELIABLE_AUROC
    try:
        auroc = roc_auc_score(y, proba)
    except ValueError:
        auroc = float('nan')
    return {
        'acc': acc, 'balanced_acc': bal_acc, 'auroc': auroc, 'auroc_reliable': auroc_reliable,
        'n_errors': n_errors, 'n_test': len(y), 'n_test_pos': n_pos, 'n_test_neg': n_neg,
    }


def fmt_metric(m):
    auroc_str = f"{m['auroc']:.4f}" + ("" if m['auroc_reliable'] else "*")
    return (f"acc {m['acc']:6.2f}%  balanced-acc {m['balanced_acc']:6.2f}%  auroc {auroc_str:>7}  "
            f"({m['n_errors']}/{m['n_test']} errors, {m['n_test_pos']} pos/{m['n_test_neg']} neg)")


def run_study_b_for_model(name, model, cam17, cam17_labels_full, device,
                          train_sel, test_sel, target_data):
    """target_data: list of dicts {spec, dataset, labels_full, sel} -- one per
    entry in Config.TARGET_DATASETS (PatchCamelyon, NCT-CRC-HE-100K, LC25000).
    The probe is fit ONCE on Camelyon17-WILDS and evaluated in-domain, then the
    SAME fitted probe (never refit) is evaluated zero-shot on every target."""
    print("\n" + "=" * 70)
    print(f"STUDY B -- {name}  (cross-dataset: Camelyon17-WILDS -> "
          f"{', '.join(td['spec']['name'] for td in target_data)})")
    print("=" * 70)
    transform = build_transform(name)

    ds_train = SinglePatchDataset(cam17, train_sel, cam17_labels_full, transform)
    ds_indomain = SinglePatchDataset(cam17, test_sel, cam17_labels_full, transform)
    loader_train = DataLoader(ds_train, batch_size=32, shuffle=False,
                              num_workers=Config.NUM_WORKERS, collate_fn=collate_single)
    loader_indomain = DataLoader(ds_indomain, batch_size=32, shuffle=False,
                                 num_workers=Config.NUM_WORKERS, collate_fn=collate_single)

    X_train, y_train = extract_masked_embeddings(model, loader_train, device)
    X_indomain, y_indomain = extract_masked_embeddings(model, loader_indomain, device)
    metrics_indomain, clf = tumor_probe_fit(X_train, y_train, X_indomain, y_indomain)
    print(f"  [in-domain -- Camelyon17->Camelyon17]   {fmt_metric(metrics_indomain)}")

    per_target = {}
    for td in target_data:
        spec = td['spec']
        ds_target = SinglePatchDataset(td['dataset'], td['sel'], td['labels_full'], transform)
        loader_target = DataLoader(ds_target, batch_size=32, shuffle=False,
                                   num_workers=Config.NUM_WORKERS, collate_fn=collate_single)
        X_target, y_target = extract_masked_embeddings(model, loader_target, device)
        metrics_target = tumor_probe_eval(clf, X_target, y_target)
        gap = metrics_indomain['balanced_acc'] - metrics_target['balanced_acc']
        print(f"  [{spec['name']:<16}]  {fmt_metric(metrics_target)}   gap {gap:+.2f} pts")
        per_target[spec['name']] = {'metrics': metrics_target, 'gap': gap}

    return {'indomain': metrics_indomain, 'targets': per_target}


# ====================================================================
# COMBINED SUMMARY -- table + figure across all 7 models
# ====================================================================
def print_study_b_summary(all_results_b):
    target_names = [d['name'] for d in Config.TARGET_DATASETS]
    print("\n" + "=" * 70)
    print(f"STUDY B SUMMARY -- cross-dataset generalization, all 7 models x {len(target_names)} target datasets")
    print("=" * 70)
    header = f"{'model':<20} {'in-domain':>10}" + "".join(f" {n[:16]:>18}" for n in target_names)
    print(header)
    for name in Config.MODEL_ORDER:
        if name not in all_results_b:
            continue
        r = all_results_b[name]
        row = f"{name:<20} {r['indomain']['balanced_acc']:>9.2f}%"
        for tn in target_names:
            if tn in r['targets']:
                row += f" {r['targets'][tn]['metrics']['balanced_acc']:>17.2f}%"
            else:
                row += f" {'--':>18}"
        print(row)
    print("\n(all balanced-accuracy; chance = 50%; in-domain = probe trained AND tested on Camelyon17-WILDS,")
    print(" never refit on any target -- every target-dataset number is a genuine zero-shot transfer)")
    for name in Config.MODEL_ORDER:
        if name not in all_results_b:
            continue
        r = all_results_b[name]
        print(f"\n{name} gaps (in-domain minus target, smaller = better transfer):")
        for tn in target_names:
            if tn in r['targets']:
                print(f"    {tn:<18} {r['targets'][tn]['gap']:>+7.2f} pts")


def make_study_b_figure(all_results_b, timestamp):
    names_b = [n for n in Config.MODEL_ORDER if n in all_results_b]
    target_names = [d['name'] for d in Config.TARGET_DATASETS]
    n_panels = len(target_names) + 1
    fig, axes = plt.subplots(1, n_panels, figsize=(5.6 * n_panels, 4.8))

    for ax, spec in zip(axes[:-1], Config.TARGET_DATASETS):
        tn = spec['name']
        xw = np.arange(len(names_b))
        indomain_vals = [all_results_b[n]['indomain']['balanced_acc'] for n in names_b]
        target_vals = [all_results_b[n]['targets'].get(tn, {}).get('metrics', {}).get('balanced_acc', np.nan)
                       for n in names_b]
        ax.bar(xw - 0.2, indomain_vals, width=0.4, label='in-domain (Camelyon17)')
        ax.bar(xw + 0.2, target_vals, width=0.4, label=tn)
        ax.axhline(50.0, ls='--', c='grey', label='chance (50%)')
        ax.set_xticks(xw); ax.set_xticklabels(names_b, rotation=35, ha='right')
        ax.set_ylabel('balanced accuracy (%)')
        ax.set_title(f"{tn}\n({spec['note'][:42]})", fontsize=9)
        ax.legend(fontsize=7); ax.grid(alpha=.3, axis='y')

    ax = axes[-1]
    width = 0.8 / len(target_names)
    xw = np.arange(len(names_b))
    for i, tn in enumerate(target_names):
        gaps = [all_results_b[n]['targets'].get(tn, {}).get('gap', np.nan) for n in names_b]
        ax.bar(xw + (i - (len(target_names) - 1) / 2) * width, gaps, width=width, label=tn)
    ax.axhline(0.0, ls='-', c='black', lw=0.8)
    ax.set_xticks(xw); ax.set_xticklabels(names_b, rotation=35, ha='right')
    ax.set_ylabel('generalization gap (pts)')
    ax.set_title('In-domain minus target, all 3 datasets\n(smaller = better transfer)', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=.3, axis='y')

    plt.tight_layout()
    out_b = f"{Config.RESULTS_DIR}/study_b_cross_dataset_all7_{timestamp}.png"
    plt.savefig(out_b, dpi=140)
    plt.close(fig)
    print(f"OK {out_b}")


def save_study_b_csv(all_results_b, timestamp):
    import pandas as pd
    # One row per (model, target dataset) -- long format, easiest to pivot in the paper.
    rows_b = []
    for name in Config.MODEL_ORDER:
        if name not in all_results_b:
            continue
        r = all_results_b[name]
        for spec in Config.TARGET_DATASETS:
            tn = spec['name']
            if tn not in r['targets']:
                continue
            m = r['targets'][tn]['metrics']
            rows_b.append({
                'model': name, 'target_dataset': tn, 'target_note': spec['note'],
                'indomain_acc': r['indomain']['acc'], 'indomain_balanced_acc': r['indomain']['balanced_acc'],
                'indomain_auroc': r['indomain']['auroc'], 'indomain_auroc_reliable': r['indomain']['auroc_reliable'],
                'target_acc': m['acc'], 'target_balanced_acc': m['balanced_acc'],
                'target_auroc': m['auroc'], 'target_auroc_reliable': m['auroc_reliable'],
                'gap_balanced_acc': r['targets'][tn]['gap'],
            })
    if rows_b:
        path_b = f"{Config.RESULTS_DIR}/study_b_cross_dataset_all7_{timestamp}.csv"
        pd.DataFrame(rows_b).to_csv(path_b, index=False)
        print(f"OK {path_b}")


# ====================================================================
# MAIN
# ====================================================================
def main():
    from datetime import datetime
    set_seed(Config.SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("=" * 70)
    print("STUDY B ONLY -- CROSS-DATASET GENERALIZATION, ALL 7 MODELS")
    print("=" * 70)
    print(f"device: {device}")
    print(f"models: {Config.MODEL_ORDER}")
    print("=" * 70)

    # ---- resolve recon checkpoints once (shared by all 3 windowed models) ----
    recon_root = _resolve_recon_checkpoint_dir()
    recon_ckpts = _find_recon_checkpoints(recon_root)
    missing_recon = [n for n in Config.WINDOWED_MODELS if n not in recon_ckpts]
    if missing_recon:
        print(f"FLAG missing recon checkpoints for {missing_recon} -- these will be SKIPPED.")

    # ---- datasets (loaded once, shared by every model) ----
    print("\nLoading Camelyon17-WILDS...")
    cam17 = load_camelyon17()
    cam17_label_col = resolve_label_column(cam17, "Camelyon17-WILDS", Config.CAMELYON17_LABEL_COLUMN)

    print("\nBuilding Camelyon17-WILDS patient-level split (for Study B's in-domain baseline)...")
    cam17_meta = cam17.select_columns(['patient', 'center', cam17_label_col]).to_pandas()
    train_pats_by_centre, test_pats_by_centre = build_patient_split_by_centre(
        cam17_meta, Config.VAL_PATIENT_FRAC, Config.SEED)
    train_mask = np.zeros(len(cam17_meta), dtype=bool)
    test_mask = np.zeros(len(cam17_meta), dtype=bool)
    for centre in Config.ALL_CENTERS:
        c_mask = (cam17_meta['center'] == centre).to_numpy()
        pat_arr = cam17_meta['patient'].to_numpy()
        train_mask |= c_mask & np.isin(pat_arr, list(train_pats_by_centre.get(centre, set())))
        test_mask |= c_mask & np.isin(pat_arr, list(test_pats_by_centre.get(centre, set())))
    cam17_labels_full = cam17_meta[cam17_label_col].to_numpy().astype(np.int64)
    train_pool_idx = np.where(train_mask)[0]
    test_pool_idx = np.where(test_mask)[0]
    train_sel = train_pool_idx[stratified_sample_indices(
        cam17_labels_full[train_pool_idx], Config.CAMELYON17_TRAIN_N_PER_CLASS, Config.SEED)]
    test_sel = test_pool_idx[stratified_sample_indices(
        cam17_labels_full[test_pool_idx], Config.CAMELYON17_TEST_N_PER_CLASS, Config.SEED + 1)]
    print(f"  Study B probe-train sample: {len(train_sel):,} patches   "
          f"in-domain test sample: {len(test_sel):,} patches")

    print("\nLoading Study B external target datasets "
          f"({len(Config.TARGET_DATASETS)}: "
          f"{', '.join(td['name'] for td in Config.TARGET_DATASETS)})...")
    target_data = []
    for i, spec in enumerate(Config.TARGET_DATASETS):
        print(f"\n--- {spec['name']} ---")
        ds = load_target_dataset(spec)
        labels_full = resolve_binary_label(
            ds, spec['label_col'], spec['positive_keywords'], spec['name'])
        sel = stratified_sample_indices(labels_full, spec['n_per_class'], Config.SEED + 2 + i)
        print(f"  Study B cross-dataset test sample ({spec['name']}): {len(sel):,} patches")
        target_data.append(dict(spec=spec, dataset=ds, labels_full=labels_full, sel=sel))

    all_results_b = {}
    for name in Config.MODEL_ORDER:
        if name in Config.WINDOWED_MODELS and name not in recon_ckpts:
            print(f"\nSKIPPING {name} (checkpoint not found)")
            continue
        try:
            model = load_model_for_probes(name, device, recon_ckpts)
        except Exception as e:
            print(f"\nFLAG could not load {name}: {e} -- SKIPPING Study B for this model.")
            continue

        try:
            all_results_b[name] = run_study_b_for_model(
                name, model, cam17, cam17_labels_full, device,
                train_sel, test_sel, target_data)
        except Exception as e:
            print(f"FLAG Study B failed for {name}: {e}")

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print_study_b_summary(all_results_b)
    make_study_b_figure(all_results_b, timestamp)
    save_study_b_csv(all_results_b, timestamp)

    print("\n" + "=" * 70)
    print("DONE.")
    print(f"   Figure + table : {Config.RESULTS_DIR}")
    print("=" * 70)
    return all_results_b


if __name__ == '__main__':
    main()

STUDY B ONLY -- CROSS-DATASET GENERALIZATION, ALL 7 MODELS
device: cuda
models: ['recon_resnet18', 'recon_vit_small', 'recon_swin_tiny', 'baseline_stain_aug', 'kaiko_vits16', 'lunit_vits16', 'phikon']
OK found recon checkpoints at classic input mount /kaggle/input/datasets/peesarisathvikreddy/histopathology-reconstruction-checkpoints

Loading Camelyon17-WILDS...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00014.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

data/train-00001-of-00014.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

data/train-00002-of-00014.parquet:   0%|          | 0.00/489M [00:00<?, ?B/s]

...

OK [phikon] loaded owkin/phikon (dim=768)

STUDY B -- phikon  (cross-dataset: Camelyon17-WILDS -> PatchCamelyon, NCT-CRC-HE-100K, LC25000)


  [in-domain -- Camelyon17->Camelyon17]   acc  86.95%  balanced-acc  86.95%  auroc  0.9635  (261/2000 errors, 1000 pos/1000 neg)


  [PatchCamelyon   ]  acc  82.15%  balanced-acc  82.15%  auroc  0.9276  (1071/6000 errors, 3000 pos/3000 neg)   gap +4.80 pts


  [NCT-CRC-HE-100K ]  acc  51.19%  balanced-acc  51.19%  auroc  0.5146  (781/1600 errors, 800 pos/800 neg)   gap +35.76 pts


  [LC25000         ]  acc  91.83%  balanced-acc  91.83%  auroc  0.9791  (327/4000 errors, 2000 pos/2000 neg)   gap -4.88 pts

STUDY B SUMMARY -- cross-dataset generalization, all 7 models x 3 target datasets
model                 in-domain      PatchCamelyon    NCT-CRC-HE-100K            LC25000
recon_resnet18           78.70%             66.43%             73.75%             60.00%
recon_vit_small          80.20%             72.92%             77.56%             76.83%
recon_swin_tiny          83.25%             73.73%             68.81%             57.53%
baseline_stain_aug       99.25%             78.62%             75.37%             69.47%
kaiko_vits16             87.70%             75.17%             63.31%             60.65%
lunit_vits16             85.90%             77.35%             83.88%             63.40%
phikon                   86.95%             82.15%             51.19%             91.83%

(all balanced-accuracy; chance = 50%; in-domain = probe trained AND tested on C